# Step 5: identify, compare and prepare for deposition

Step 3 produced a consensus genome for each run. This notebook asks what it is, how it compares with other records across borders, and prepares it for deposition back into a public archive.

What it does, in order:

1. **Nextclade** reads each consensus and reports its clade and Pango lineage, plus a quality verdict.
2. It compares our consensus with the one **ENA's own pipeline** made from the same reads.
3. It searches for the closest matches in the **COVID-19 Data Portal** (EMBL-EBI), using BLAST.
4. It pulls **Swedish** sequences from ENA, runs the same Nextclade on them and sets them next to the Irish ones.
5. It collects the **metadata** a deposition needs, from the record saved in step 2, and lets you correct it by hand.
6. It builds the **submission file** for ENA's SARS-CoV-2 genome service.
7. Optionally it checks that file against ENA's **test** service.

**This notebook never submits anything to ENA's real service.** These reads belong to the Irish National Virus Reference Laboratory, and ENA already holds a consensus for them. Everything in sections 5 to 7 is a worked example with dummy values, not a real submission.

Run the cells one at a time, from top to bottom, and read the output of each before moving on.

## 1. Imports

- `csv` reads and writes tab-separated tables.
- `gzip` unpacks compressed files.
- `json` reads and writes JSON, a format for structured data.
- `getpass` asks you to type a password without showing it.
- `subprocess` runs another program from Python.
- `time` lets the notebook wait.
- `os`, `sys` and `shutil` are used once, to check the tools can be found.
- `Path` builds file and folder paths.
- `requests` fetches web addresses.

In [ ]:
import csv
import getpass
import gzip
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests

## 2. Your input and the folders

- `ACCESSION`: the same as in the earlier steps.
- `EMAIL`: the BLAST service in section 5 asks for an email address, so EMBL-EBI can contact you if there is a problem. Type yours between the quotes. If you leave it empty, section 5 is skipped.

The notebook makes one new folder, `deposition/`, for everything it writes.

In [ ]:
ACCESSION = "PRJEB40277"
EMAIL = ""

DATA_FOLDER = Path("data") / ACCESSION
METADATA_FOLDER = DATA_FOLDER / "metadata"
MAPPING_FOLDER = DATA_FOLDER / "mapping"
DEPOSITION_FOLDER = DATA_FOLDER / "deposition"
REFERENCE_FOLDER = Path("data") / "reference"

DEPOSITION_FOLDER.mkdir(parents=True, exist_ok=True)

consensus_files = sorted(MAPPING_FOLDER.glob("*.consensus.fasta"))

for consensus_file in consensus_files:
    print(consensus_file.name)

## 3. Check the tool can be found

The first line makes sure this notebook can find programs installed in the `ls-use-case` environment.

**Look at the output.** The line should show a location that contains `ls-use-case`. If you see `None`, choose the `ls-use-case` kernel (top right) and run the cell again.

In [ ]:
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

print("nextclade ->", shutil.which("nextclade"))

## 4. Nextclade: clade and lineage

[Nextclade](https://clades.nextstrain.org/) compares a genome with a reference tree and reports which **clade** (Nextstrain's naming, for example `21J`) and which **Pango lineage** (the naming used worldwide, for example `AY.4`) it belongs to. It also gives a quality verdict.

Two steps:

1. `nextclade dataset get` downloads the SARS-CoV-2 dataset: the reference and the tree. It is small. If it is already on your disk it is skipped.
2. `nextclade run` reads a FASTA file and writes a table of results. All the consensus files are first joined into one FASTA file, so Nextclade is run once.

The table is tab-separated. The cell reads it and prints a few columns:

- `clade` and `Nextclade_pango`: the clade and the Pango lineage.
- `qc.overallStatus`: `good`, `mediocre` or `bad`.
- `totalMissing`: the number of unknown bases (`N`).
- `totalSubstitutions`: how many single-base differences there are from the reference.

Lines worth explaining:

- `csv.DictReader` reads a table so each row can be asked for a column by name.
- Nextclade names each sequence in its results by the whole name line. Ours are long, so the cell writes the combined file with just the run accession as the name.
- `consensus_file.name.split(".")[0]` is the run accession, the part of the file name before the first `.`.
- `lines[1:]` is every line after the name line, and `"".join(...)` glues them into one sequence.

**Look at the output.** Nextclade takes a few seconds.

In [ ]:
dataset_folder = REFERENCE_FOLDER / "nextclade_sars-cov-2"

if dataset_folder.exists():
    print("Already here, skipping:", dataset_folder)
else:
    subprocess.run(["nextclade", "dataset", "get", "--name", "sars-cov-2", "--output-dir", str(dataset_folder)], check=True)

all_consensus_file = DEPOSITION_FOLDER / "consensus_all_runs.fasta"

with open(all_consensus_file, "w") as output_file:
    for consensus_file in consensus_files:
        run = consensus_file.name.split(".")[0]
        lines = consensus_file.read_text().strip().split("\n")
        sequence = "".join(lines[1:])
        output_file.write(f">{run}\n{sequence}\n")

irish_table = DEPOSITION_FOLDER / "nextclade_irish.tsv"

subprocess.run(
    ["nextclade", "run", "--input-dataset", str(dataset_folder), "--output-tsv", str(irish_table), str(all_consensus_file)],
    check=True,
    stderr=subprocess.DEVNULL,
)

print()
with open(irish_table) as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        print(row["seqName"], "clade:", row["clade"], " lineage:", row["Nextclade_pango"],
              " quality:", row["qc.overallStatus"], " N:", row["totalMissing"], " substitutions:", row["totalSubstitutions"])

## 5. Compare with ENA's own consensus

ENA's pipeline has already produced a consensus from these reads, with its own trimming and its own caller. If our result agrees with it, that is a good sign for both.

ENA's consensus files sit on its FTP server at an address that includes an **analysis accession** (it starts with `ERZ`). Looking that accession up automatically needs the ENA Portal API, which was not working when this notebook was written. So the address of the one run we checked is typed in below.

To find the address for another run, open the run's page at ENA, for example <https://www.ebi.ac.uk/ena/browser/view/ERR7112279>, and look for the analysis linked to it. Then add one line to `ENA_CONSENSUS_URLS`.

The cell downloads each file, unpacks it, saves it, and runs Nextclade on it. Then it prints our result and ENA's side by side. It also prints which substitutions only one of the two has.

Lines worth explaining:

- `gzip.decompress(...)` unpacks the downloaded `.gz` bytes.
- The name line is replaced by the run accession plus `_ENA`, so the two can be told apart.
- `set(...)` holds each item once. Subtracting one set from another leaves the items only the first one has. Here the items are the individual substitutions, such as `C241T`.

**Look at the output.** The clade and lineage should be the same. The number of `N` and the list of substitutions can differ by a few bases, because the two pipelines choose depth cut-offs, trimming and consensus rules a little differently. A difference of one or two bases is normal. The lines at the bottom say exactly which ones.

In [ ]:
ENA_CONSENSUS_URLS = {
    "ERR7112279": "https://ftp.sra.ebi.ac.uk/vol1/analysis/ERZ765/ERZ7651362/ERR7112279_consensus.fasta.gz",
}

ena_fasta = DEPOSITION_FOLDER / "ena_consensus_all_runs.fasta"

with open(ena_fasta, "w") as output_file:
    for run, url in ENA_CONSENSUS_URLS.items():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        text = gzip.decompress(response.content).decode()
        lines = text.strip().split("\n")
        sequence = "".join(lines[1:])
        output_file.write(f">{run}_ENA\n{sequence}\n")

ena_table = DEPOSITION_FOLDER / "nextclade_ena.tsv"

subprocess.run(
    ["nextclade", "run", "--input-dataset", str(dataset_folder), "--output-tsv", str(ena_table), str(ena_fasta)],
    check=True,
    stderr=subprocess.DEVNULL,
)

ours = {}
with open(irish_table) as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        ours[row["seqName"]] = row

print()
with open(ena_table) as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        run = row["seqName"].replace("_ENA", "")
        mine = ours[run]
        print(run)
        print("    this use case:  clade", mine["clade"], " lineage", mine["Nextclade_pango"], " N:", mine["totalMissing"], " substitutions:", mine["totalSubstitutions"])
        print("    ENA pipeline:   clade", row["clade"], " lineage", row["Nextclade_pango"], " N:", row["totalMissing"], " substitutions:", row["totalSubstitutions"])
        ours_list = set(mine["substitutions"].split(","))
        ena_list = set(row["substitutions"].split(","))
        print("    only in this use case:", sorted(ours_list - ena_list))
        print("    only in ENA pipeline: ", sorted(ena_list - ours_list))

## 6. Closest matches in the COVID-19 Data Portal

The [COVID-19 Data Portal](https://www.covid19dataportal.org/) (EMBL-EBI) holds consensus sequences made by ENA's pipeline for SARS-CoV-2 reads from many countries. EMBL-EBI offers BLAST over this collection as a web service. BLAST finds the stored sequences most similar to the one you send.

This cell sends the **first** consensus only, because each search takes a few minutes.

How the service works: you submit a job and get a job ID back. You then ask for its status until it says `FINISHED`, and then ask for the result.

- `database` is `cdp`, the COVID-19 Data Portal consensus sequences.
- `program` is `blastn`, BLAST for DNA against DNA.
- `time.sleep(10)` waits 10 seconds between status checks.

The result is a table. Each hit has an accession, a description, and the percentage of identical bases. The first line of the table is the column names, so the cell skips it with `[1:]`, which means "everything after the first item".

**Look at the output.** The descriptions name the country and date. Many Delta-lineage genomes from different places are identical or nearly so, so expect several hits at or near 100%. BLAST shows which records are close, not where this sample came from.

If `EMAIL` at the top is empty, this cell does nothing.

In [ ]:
if EMAIL == "":
    print("EMAIL is empty, skipping the BLAST search.")
else:
    BLAST_URL = "https://www.ebi.ac.uk/Tools/services/rest/ncbiblast/"

    first_consensus = consensus_files[0]

    settings = {
        "email": EMAIL,
        "program": "blastn",
        "stype": "dna",
        "database": "cdp",
        "alignments": "10",
        "scores": "10",
        "sequence": first_consensus.read_text(),
    }

    response = requests.post(BLAST_URL + "run", data=settings, timeout=60)
    response.raise_for_status()
    job_id = response.text.strip()
    print("Searching with", first_consensus.name, " job:", job_id)

    status = ""
    while status != "FINISHED":
        time.sleep(10)
        status = requests.get(BLAST_URL + "status/" + job_id, timeout=60).text.strip()
        print("    status:", status)
        if status == "FAILED":
            break

    if status == "FINISHED":
        result = requests.get(BLAST_URL + "result/" + job_id + "/tsv", timeout=60).text
        for line in result.strip().split("\n")[1:]:
            columns = line.split("\t")
            print(columns[2], " identity:", columns[7], "%  ", columns[3][:90])

## 7. Cross-border: Swedish sequences

Now the same method on data from another country. The aim is to run the **same** Nextclade on Swedish sequences and set them next to the Irish ones.

The Swedish Pathogens Portal (<https://www.pathogens.se>) does not offer a tool or an API for this. It tells data providers to deposit in ENA, and Swedish sequences are therefore in ENA. So this cell asks ENA for them.

Two services are used:

- **EBI Search** lists records. The query `country:Sweden AND collection_date:2021*` means: collected in Sweden, in 2021. `size` is how many to return.
- **ENA Browser API** gives the sequence of each record as FASTA.

The cell downloads five sequences, runs Nextclade on them, and prints their lineage next to the Irish result from section 4.

**What this does and does not show.** ENA holds hardly any Swedish SARS-CoV-2 sequences from the same weeks as the Irish samples (September 2021), so this is **not** a like-for-like comparison. The Swedish sequences here are mostly from early 2021, when other lineages were circulating. It shows the method: any country's sequences can be pulled from ENA and put through the same tool.

**Look at the output.**

In [ ]:
SEARCH_URL = "https://www.ebi.ac.uk/ebisearch/ws/rest/embl-covid19"

settings = {
    "query": "country:Sweden AND collection_date:2021*",
    "fields": "country,collection_date,isolate",
    "format": "json",
    "size": "5",
}

response = requests.get(SEARCH_URL, params=settings, timeout=60)
response.raise_for_status()
entries = response.json()["entries"]

swedish_fasta = DEPOSITION_FOLDER / "swedish_sequences.fasta"
collection_dates = {}

with open(swedish_fasta, "w") as output_file:
    for entry in entries:
        accession = entry["id"]
        collection_dates[accession] = entry["fields"]["collection_date"][0]

        response = requests.get("https://www.ebi.ac.uk/ena/browser/api/fasta/" + accession, timeout=60)
        response.raise_for_status()
        lines = response.text.strip().split("\n")
        sequence = "".join(lines[1:])
        output_file.write(f">{accession}\n{sequence}\n")

swedish_table = DEPOSITION_FOLDER / "nextclade_swedish.tsv"

subprocess.run(
    ["nextclade", "run", "--input-dataset", str(dataset_folder), "--output-tsv", str(swedish_table), str(swedish_fasta)],
    check=True,
    stderr=subprocess.DEVNULL,
)

print()
print("IRELAND")
for run, row in ours.items():
    print("   ", run, "collected", "-", " clade", row["clade"], " lineage", row["Nextclade_pango"])

print("SWEDEN")
with open(swedish_table) as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        print("   ", row["seqName"], "collected", collection_dates[row["seqName"]], " clade", row["clade"], " lineage", row["Nextclade_pango"])

## 8. Metadata for deposition

A sequence with no description is not worth depositing. ENA asks for a set of fields about the sample. For SARS-CoV-2 and other viruses the set is the checklist **ERC000033**. It has 10 mandatory fields:

collection date, country, host common name, host subject id, host health state, host sex, host scientific name, collector name, collecting institution, isolate.

(The full checklist is at <https://www.ebi.ac.uk/ena/browser/api/xml/ERC000033>.)

These runs are already in ENA with their sample records. In step 2 we saved those records, so the values can be taken from `metadata.tsv`. For a real new sample, they would come from your lab records instead.

Two ways to set a value:

- **Automatic**: the cell copies the field from `metadata.tsv`.
- **By hand**: add the run and the field to `OVERRIDES` below. It wins over the automatic value.

Every value is printed with where it came from: `ENA record`, `typed by hand`, or `MISSING` if nothing could be found.

A field ENA allows to be unknown is written as `not provided`. That is accepted by ENA, and is different from an empty value, which is not.

Lines worth explaining:

- `OVERRIDES.get(run, {})` gives the corrections for this run, or an empty set of corrections if there are none.
- A field in `metadata.tsv` has the prefix `sample: `, so the cell adds it when looking up.

The table is saved in `deposition/deposition_metadata.tsv`.

**Look at the output.** Read every line. A value you are not sure about should not be deposited.

In [ ]:
MANDATORY_FIELDS = [
    "collection date",
    "geographic location (country and/or sea)",
    "host common name",
    "host subject id",
    "host health state",
    "host sex",
    "host scientific name",
    "collector name",
    "collecting institution",
    "isolate",
]

OVERRIDES = {
    # "ERR7112279": {"collector name": "Example Person"},
}

metadata_rows = {}
with open(METADATA_FOLDER / "metadata.tsv") as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        metadata_rows[row["run_accession"]] = row

deposition_rows = []

for consensus_file in consensus_files:
    run = consensus_file.name.split(".")[0]
    metadata = metadata_rows[run]
    corrections = OVERRIDES.get(run, {})

    deposition_row = {"run": run}
    typed_by_hand = []

    print(run)
    for field in MANDATORY_FIELDS:
        if field in corrections:
            value = corrections[field]
            source = "typed by hand"
            typed_by_hand.append(field)
        else:
            value = metadata.get("sample: " + field, "")
            source = "ENA record"

        if value == "":
            source = "MISSING"

        deposition_row[field] = value
        print("   ", field, "=", value, " [" + source + "]")

    deposition_row["typed_by_hand"] = ", ".join(typed_by_hand)
    deposition_rows.append(deposition_row)

columns = ["run"] + MANDATORY_FIELDS + ["typed_by_hand"]

with open(DEPOSITION_FOLDER / "deposition_metadata.tsv", "w", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=columns, delimiter="\t")
    writer.writeheader()
    writer.writerows(deposition_rows)

print()
print("Written:", DEPOSITION_FOLDER / "deposition_metadata.tsv")

## 9. Build the submission file

ENA has a web service for submitting SARS-CoV-2 genomes. It takes a JSON file with:

| Field | Meaning |
|---|---|
| `name` | A unique name for this assembly |
| `study` | The ENA study (project) it belongs to |
| `sample` | The ENA sample it came from |
| `coverage` | The average depth of the reads |
| `program` | The software that made the sequence |
| `platform` | The sequencing instrument |
| `sequence` | The genome sequence |

**These values are dummy values for the example.** `DEPOSIT_STUDY` must be a study that **you** own in ENA. The real study `PRJEB40277` belongs to the Irish consortium, so it is not used. Until you change it, the notebook uses a placeholder that cannot be submitted.

`coverage` is the mean depth from step 3. `platform` is built from the instrument model in the metadata. The exact wording ENA accepts for `program` and `platform` has not been checked against the service. Section 10 tells you whether it accepts them.

One file per run is written to `deposition/<run>.genome_submission.json`. The sequence is long, so the cell prints everything except the sequence.

Lines worth explaining:

- `json.dump(..., indent=2)` writes the JSON with line breaks and indentation, so a person can read it.
- The cell prints the submission before adding the sequence to it, so the printout stays short.

**Look at the output.**

In [ ]:
DEPOSIT_STUDY = "PRJEBXXXXX"
PROGRAM = "minimap2 2.31, samtools 1.24"

mean_depths = {}
with open(MAPPING_FOLDER / "summary.tsv") as input_file:
    for row in csv.DictReader(input_file, delimiter="\t"):
        mean_depths[row["run"]] = round(float(row["mean_depth"]))

submissions = {}

for consensus_file in consensus_files:
    run = consensus_file.name.split(".")[0]
    metadata = metadata_rows[run]

    lines = consensus_file.read_text().strip().split("\n")
    sequence = "".join(lines[1:])

    submission = {}
    submission["name"] = f"{run}_demo_consensus"
    submission["study"] = DEPOSIT_STUDY
    submission["sample"] = metadata["sample_accession"]
    submission["coverage"] = mean_depths[run]
    submission["program"] = PROGRAM
    submission["platform"] = "Oxford Nanopore " + metadata["instrument_model"]

    print(json.dumps(submission, indent=2))
    print("sequence:", len(sequence), "bases")
    print()

    submission["sequence"] = sequence
    submissions[run] = submission

    with open(DEPOSITION_FOLDER / f"{run}.genome_submission.json", "w") as output_file:
        json.dump(submission, output_file, indent=2)

if DEPOSIT_STUDY == "PRJEBXXXXX":
    print("DEPOSIT_STUDY is still the placeholder. The files cannot be submitted until you change it.")

## 10. Optional: check the file against ENA's test service

ENA runs a **test** copy of its submission service. A file sent there is checked and then thrown away within 24 hours. It is never published. This cell sends the **first** run's file to the test service's **validate** address, which only checks it.

You need a **Webin account**, which is ENA's submission login. If you do not have one, leave `WEBIN_USER` empty and skip this cell. Nothing else depends on it.

Your password is asked for when the cell runs, and is not saved in the notebook. For a validation to pass, `DEPOSIT_STUDY` must be one of your own studies, and the sample must be one your account can use. With the dummy values above, expect ENA to refuse it, and read the message: it tells you which field is wrong.

There is deliberately **no code in this notebook that submits for real**. Following the Swedish Pathogens Portal tutorial (<https://pathogens.se/support_services/tutorial_ena/tutorial_ena_intro/>) is the way to do a real submission, using the test service first.

This part of the notebook could not be tested by its author, because it needs an account. Treat the first run as a trial.

`auth=(WEBIN_USER, password)` sends the account name and password to ENA with the request.

**Look at the output.** Status `200` means ENA accepted the file as valid. Any other status comes with a message from ENA.

In [ ]:
WEBIN_USER = ""

VALIDATE_URL = "https://wwwdev.ebi.ac.uk/ena/submit/webin-cli/api/v1/genome/covid-19/validate"

if WEBIN_USER == "":
    print("WEBIN_USER is empty, skipping.")
else:
    password = getpass.getpass("Webin password: ")
    first_run = list(submissions)[0]

    response = requests.post(VALIDATE_URL, json=submissions[first_run], auth=(WEBIN_USER, password), timeout=120)

    print("Run:", first_run)
    print("HTTP status:", response.status_code)
    print(response.text[:1500])

## What you have now

In `deposition/`:

- `nextclade_irish.tsv`, `nextclade_ena.tsv`, `nextclade_swedish.tsv`: clade, lineage and quality for each set of sequences.
- `deposition_metadata.tsv`: the 10 mandatory sample fields for each run, and which ones were typed by hand.
- `<run>.genome_submission.json`: the file the ENA SARS-CoV-2 genome service takes.

That is the end of the use case as scaffolded: Irish reads from ENA, to quality checks, to a consensus genome, to a lineage call, to a comparison with ENA's own result and with Swedish data, to a deposition-ready file with its metadata.